# ISAF — XGBoost evaluation

This notebook trains one final XGBoost classifier and evaluates predictive/economic performance, interpretability, stability, and group fairness.

**Target:** `loan_paid_back = 1` means repaid; `0` means default. Every predicted probability refers to repayment. Approve when `P(repaid) >= threshold`. A default-probability cutoff of 0.16 corresponds to a repayment cutoff of 0.84 (with the boundary convention stated explicitly).

**Feature switch:** `INCLUDE_EMPLOYMENT_AND_EDUCATION = True` adds `employment_status` and `education_level` to seven base predictors (nine inputs total). `False` uses the seven base predictors. Gender and marital status are excluded in both modes. All four attributes remain available for the fairness audit; audit membership is separate from predictor inclusion.

**Protocol:** remove shared-test IDs before sampling or fitting. Each seed uses disjoint fit / early-stop / threshold-tuning / independent-validation partitions. Select the final seed in advance, and select the threshold using only its threshold-tuning partition. Use the shared test for final reporting, never for choosing features, parameters, seeds, or thresholds. Repeated exploratory comparisons on this shared test must be disclosed.

**Run:** restart the kernel and run all cells after changing settings. The startup banner shows the active switch, expected feature count and run directory. Each run saves to `MLModels/xgboost_runs/<feature mode>/<full or fast>/<run ID>/`; read the summary and configuration from that exact directory. Saved results from earlier runs do not update when source code changes.

## 1. Imports and experiment settings

The notebook uses XGBoost’s native categorical support. SHAP is optional at import time so predictive, economic, stability, and fairness outputs can still be produced if the local SHAP installation needs repair; permutation importance remains available in that case.

In [ ]:
from pathlib import Path
from itertools import combinations
import json
from datetime import datetime, timezone
from importlib.metadata import version
import uuid
from IPython.display import display
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score, roc_curve,
    brier_score_loss, accuracy_score, confusion_matrix
)
from sklearn.inspection import permutation_importance
from xgboost import XGBClassifier
import xgboost as xgb

try:
    import shap
    SHAP_AVAILABLE = True
    SHAP_IMPORT_ERROR = None
except Exception as exc:
    shap = None
    SHAP_AVAILABLE = False
    SHAP_IMPORT_ERROR = repr(exc)


# Find the repository from its data files, starting at the notebook working directory.
PROJECT_DIR = next((p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / "data/train.csv").is_file()
                    and (p / "data/test_eval_20k.csv").is_file()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError("Start Jupyter from the project root or MLModels directory.")
DATA_DIR = PROJECT_DIR / "data"

TRAIN_PATH = DATA_DIR / "train.csv"
EVAL_PATH = DATA_DIR / "test_eval_20k.csv"
TARGET = "loan_paid_back"  # 1 = repaid; 0 = default
ID_COL = "id"
FAIRNESS_ATTRIBUTES = ["gender", "marital_status", "employment_status", "education_level"]
CAT_COLS = [
    "gender", "marital_status", "education_level", "employment_status",
    "loan_purpose", "grade_subgrade"
]

# Team-aligned feature switch:
# True  -> use employment_status and education_level as predictors.
# False -> exclude all four audited attributes from model predictors.
# gender and marital_status remain excluded in both modes. All four attributes
# are always retained for fairness reporting and proxy-association checks.
INCLUDE_EMPLOYMENT_AND_EDUCATION = True
OPTIONAL_PREDICTORS = (
    ["employment_status", "education_level"]
    if INCLUDE_EMPLOYMENT_AND_EDUCATION else []
)
BASE_FEATURES = ["annual_income", "debt_to_income_ratio", "credit_score",
                 "loan_amount", "interest_rate", "loan_purpose", "grade_subgrade"]
FEATURE_COLS = BASE_FEATURES + OPTIONAL_PREDICTORS
assert set(OPTIONAL_PREDICTORS).issubset(FAIRNESS_ATTRIBUTES)
assert not set(OPTIONAL_PREDICTORS).intersection({"gender", "marital_status"})

RUN_LABEL = (
    "with_employment_education"
    if INCLUDE_EMPLOYMENT_AND_EDUCATION
    else "sensitive_features_excluded"
)

# Full pool is the final setting. Flip to True only for a quick code check.
FAST_MODE = False
FAST_SAMPLE_SIZE = 100_000

# A four-way split keeps fitting, early stopping, threshold selection, and
# stability evaluation on disjoint rows. The final shared test stays untouched.
RANDOM_STATE = 42
STABILITY_SEEDS = [1, 10, 42, 100, 2026]
FIT_FRACTION = 0.70
EARLY_STOP_FRACTION = 0.10
THRESHOLD_FRACTION = 0.10
STABILITY_FRACTION = 0.10

# Unweighted fitting is the default. Class weighting changes probability calibration;
# if enabled, derive weights from the fit partition only and label the run.
USE_CLASS_WEIGHT = False

N_ESTIMATORS = 1500
EARLY_STOPPING_ROUNDS = 50
THRESHOLDS = np.arange(0.01, 1.00, 0.01)
TOST_DELTA = 0.02  # equivalence margin for approval-rate differences (2 pp)
TOST_ALPHA = 0.05
MIN_GROUP_N_FOR_TOST = 30
SHAP_SAMPLE_SIZE = 1000
PERMUTATION_REPEATS = 5

assert RANDOM_STATE in STABILITY_SEEDS, "The final seed must be in STABILITY_SEEDS."
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
OUTPUT_DIR = PROJECT_DIR / "MLModels" / "xgboost_runs" / RUN_LABEL / ("fast" if FAST_MODE else "full") / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
print("Run ID:", RUN_ID)
print("Include employment and education:", INCLUDE_EMPLOYMENT_AND_EDUCATION)
print("Expected model features:", len(FEATURE_COLS), FEATURE_COLS)
print("Output directory:", OUTPUT_DIR)

## 2. Load data and verify the shared-test boundary

All 20,000 evaluation IDs must be present in the original training file and absent from the model-training pool. This assertion runs before sampling, splitting, fitting, or threshold selection.

In [ ]:
train_full = pd.read_csv(TRAIN_PATH)
eval_df = pd.read_csv(EVAL_PATH)

required = {*BASE_FEATURES, ID_COL, TARGET, "loan_amount", "interest_rate", *FAIRNESS_ATTRIBUTES, *CAT_COLS}
missing_train = required - set(train_full.columns)
missing_eval = required - set(eval_df.columns)
assert not missing_train, f"Missing training columns: {sorted(missing_train)}"
assert not missing_eval, f"Missing evaluation columns: {sorted(missing_eval)}"
assert train_full[ID_COL].is_unique, "Training IDs must be unique."
assert eval_df[ID_COL].is_unique, "Shared-test IDs must be unique."
assert train_full[TARGET].isin([0, 1]).all(), "Target must be binary 0/1."
assert eval_df[TARGET].isin([0, 1]).all(), "Evaluation target must be binary 0/1."

shared_ids = set(eval_df[ID_COL])
train_ids = set(train_full[ID_COL])
assert shared_ids.issubset(train_ids), "Some shared-test IDs are not in train.csv."
training_pool = train_full.loc[~train_full[ID_COL].isin(shared_ids)].copy()
assert not set(training_pool[ID_COL]).intersection(shared_ids), "Shared-test ID leakage detected."

assert len(eval_df) == 20_000, "Expected the shared 20,000-row evaluation set."
assert train_full[ID_COL].notna().all() and eval_df[ID_COL].notna().all()
for df in (training_pool, eval_df):
    assert np.isfinite(df[["loan_amount", "interest_rate"]].to_numpy(dtype=float)).all()
    assert (df[["loan_amount", "interest_rate"]] >= 0).all().all()

if FAST_MODE and len(training_pool) > FAST_SAMPLE_SIZE:
    sample_idx, _ = train_test_split(
        training_pool.index,
        train_size=min(FAST_SAMPLE_SIZE, len(training_pool)),
        stratify=training_pool[TARGET],
        random_state=RANDOM_STATE,
    )
    training_pool = training_pool.loc[sample_idx].copy()

assert FEATURE_COLS, "No model features remain."
assert not set(FEATURE_COLS).intersection({ID_COL, TARGET})
assert set(OPTIONAL_PREDICTORS).issubset(FEATURE_COLS)
assert not set(["gender", "marital_status"]).intersection(FEATURE_COLS)

print(f"Original train rows: {len(train_full):,}")
print(f"Shared test rows:    {len(eval_df):,}")
print(f"Training pool rows:  {len(training_pool):,}")
print(f"ID overlap:          {len(set(training_pool[ID_COL]) & set(eval_df[ID_COL]))}")
print(f"Model features ({len(FEATURE_COLS)}): {FEATURE_COLS}")
print(f"Audited attributes used as predictors: {OPTIONAL_PREDICTORS or 'None'}")
print(f"Fast mode: {FAST_MODE}")
print(f"Repayment rate in training pool: {training_pool[TARGET].mean():.4f}")

## 3. Economic rule and reusable evaluation helpers

Approval means the estimated probability of repayment is at least the selected threshold. The threshold is chosen by maximizing validation P&L only. The shared-test P&L is calculated after that choice and is never used to tune the policy.

In [ ]:
def profit_vector(y_true, approved, loan_amount, interest_rate):
    """Course rule: repaid approval earns interest; default loses 60% of principal; reject earns 0."""
    y_true = np.asarray(y_true, dtype=int)
    approved = np.asarray(approved, dtype=bool)
    loan_amount = np.asarray(loan_amount, dtype=float)
    interest_rate = np.asarray(interest_rate, dtype=float)
    return np.where(
        approved & (y_true == 1), loan_amount * interest_rate / 100.0,
        np.where(approved & (y_true == 0), -loan_amount * 0.60, 0.0),
    )


def select_threshold(y_true, probabilities, loan_amount, interest_rate):
    rows = []
    y_arr = np.asarray(y_true, dtype=int)
    p_arr = np.asarray(probabilities, dtype=float)
    amount = np.asarray(loan_amount, dtype=float)
    rate = np.asarray(interest_rate, dtype=float)
    for threshold in THRESHOLDS:
        approved = p_arr >= threshold
        pnl = profit_vector(y_arr, approved, amount, rate).sum()
        rows.append({
            "Threshold": float(threshold),
            "P&L": float(pnl),
            "P&L per Applicant": float(pnl / len(y_arr)),
            "Approval Rate": float(approved.mean()),
        })
    table = pd.DataFrame(rows)
    best_row = table.loc[table["P&L"].idxmax()]
    return float(best_row["Threshold"]), table


def evaluate_predictions(y_true, probabilities, loan_amount, interest_rate, threshold):
    y_arr = np.asarray(y_true, dtype=int)
    p_arr = np.asarray(probabilities, dtype=float)
    decisions = (p_arr >= threshold).astype(int)
    fpr, tpr, _ = roc_curve(y_arr, p_arr)
    tn, fp, fn, tp = confusion_matrix(y_arr, decisions, labels=[0, 1]).ravel()
    pnl = profit_vector(y_arr, decisions, loan_amount, interest_rate).sum()
    approved_count = int(decisions.sum())
    return {
        "Applicants": len(y_arr),
        "ROC-AUC": float(roc_auc_score(y_arr, p_arr)),
        "PR-AUC": float(average_precision_score(y_arr, p_arr)),
        "Gini": float(2 * roc_auc_score(y_arr, p_arr) - 1),
        "KS": float(np.max(np.abs(tpr - fpr))),
        "Brier Score": float(brier_score_loss(y_arr, p_arr)),
        "PCC / Accuracy": float(accuracy_score(y_arr, decisions)),
        "Threshold": float(threshold),
        "P&L": float(pnl),
        "P&L per Applicant": float(pnl / len(y_arr)),
        "Approval Rate": float(decisions.mean()),
        "Default Rate Among Approved": float((y_arr[decisions == 1] == 0).mean()) if approved_count else np.nan,
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
    }


def split_training_pool(seed):
    """Return fit, early-stop, threshold, and independent stability partitions."""
    fractions = np.array([FIT_FRACTION, EARLY_STOP_FRACTION, THRESHOLD_FRACTION, STABILITY_FRACTION])
    assert np.isclose(fractions.sum(), 1.0), "The four split fractions must sum to 1."
    assert np.all(fractions > 0), "Each split fraction must be positive."
    X = training_pool[FEATURE_COLS].copy()
    y = training_pool[TARGET].astype(int).copy()
    X_fit, X_rest, y_fit, y_rest = train_test_split(
        X, y, test_size=1 - FIT_FRACTION, stratify=y, random_state=seed
    )
    # Divide the remaining rows according to the configured global fractions.
    early_stop_share_of_rest = EARLY_STOP_FRACTION / (1 - FIT_FRACTION)
    X_es, X_rest2, y_es, y_rest2 = train_test_split(
        X_rest, y_rest, test_size=1 - early_stop_share_of_rest,
        stratify=y_rest, random_state=seed + 1
    )
    threshold_share_of_remainder = THRESHOLD_FRACTION / (THRESHOLD_FRACTION + STABILITY_FRACTION)
    X_thr, X_stab, y_thr, y_stab = train_test_split(
        X_rest2, y_rest2, test_size=1 - threshold_share_of_remainder,
        stratify=y_rest2, random_state=seed + 2
    )
    # Learn category vocabularies only from fit rows. Unseen levels become missing
    # for the model; raw audit attributes stay intact in training_pool/eval_df.
    for col in [c for c in CAT_COLS if c in FEATURE_COLS]:
        dtype = pd.CategoricalDtype(categories=sorted(X_fit[col].dropna().unique()))
        for frame in (X_fit, X_es, X_thr, X_stab):
            frame[col] = frame[col].astype(dtype)
    partitions = [set(frame.index) for frame in (X_fit, X_es, X_thr, X_stab)]
    assert sum(map(len, partitions)) == len(set.union(*partitions)) == len(training_pool)
    return X_fit, y_fit, X_es, y_es, X_thr, y_thr, X_stab, y_stab


def build_model(seed, fit_target):
    positives = int((fit_target == 1).sum())
    negatives = int((fit_target == 0).sum())
    scale_pos_weight = negatives / positives if USE_CLASS_WEIGHT else 1.0
    return XGBClassifier(
        objective="binary:logistic",
        eval_metric="auc",
        n_estimators=N_ESTIMATORS,
        max_depth=5,
        learning_rate=0.05,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.5,
        reg_lambda=5.0,
        scale_pos_weight=scale_pos_weight,
        enable_categorical=True,
        tree_method="hist",
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        random_state=seed,
        n_jobs=-1,
    )

## 4. Train the final XGBoost model and measure stability

For each seed, four disjoint partitions are used: 70% to fit, 10% for early stopping, 10% to select the economic threshold, and 10% for an independent stability score. Seed 42 is the final model; the other seeds measure sensitivity to data partition and random initialization. No seed is chosen based on shared-test performance.

In [ ]:
models = {}
splits = {}
stability_rows = []

for seed in STABILITY_SEEDS:
    (X_fit, y_fit, X_es, y_es, X_thr, y_thr, X_stab, y_stab) = split_training_pool(seed)
    model = build_model(seed, y_fit)
    model.fit(
        X_fit, y_fit,
        eval_set=[(X_es, y_es)],
        verbose=False,
    )

    # Tune the operating threshold on its own held-out partition.
    p_thr = model.predict_proba(X_thr)[:, 1]
    threshold, _ = select_threshold(
        y_thr, p_thr,
        training_pool.loc[X_thr.index, "loan_amount"],
        training_pool.loc[X_thr.index, "interest_rate"],
    )

    # Score stability on a separate partition that did not select the threshold.
    p_stab = model.predict_proba(X_stab)[:, 1]
    stab_metrics = evaluate_predictions(
        y_stab, p_stab,
        training_pool.loc[X_stab.index, "loan_amount"],
        training_pool.loc[X_stab.index, "interest_rate"],
        threshold,
    )
    stability_rows.append({
        "Seed": seed,
        "Best Iteration": int(model.best_iteration) if model.best_iteration is not None else np.nan,
        **stab_metrics,
    })
    if seed == RANDOM_STATE:
        final_split_sizes = dict(fit=len(X_fit), early_stop=len(X_es), threshold=len(X_thr), independent_validation=len(X_stab))
        final_split_ids = pd.concat([
            pd.DataFrame({ID_COL: training_pool.loc[frame.index, ID_COL], "Partition": label})
            for label, frame in [("fit", X_fit), ("early_stop", X_es), ("threshold", X_thr), ("independent_validation", X_stab)]
        ], ignore_index=True)
    models[seed] = model
    splits[seed] = {
        "X_thr": X_thr, "y_thr": y_thr, "p_thr": p_thr,
        "X_stab": X_stab, "y_stab": y_stab,
    }
    print(f"Seed {seed}: best iteration={model.best_iteration}, threshold={threshold:.2f}, stability AUC={stab_metrics['ROC-AUC']:.4f}")

stability_runs = pd.DataFrame(stability_rows)
stability_summary = (
    stability_runs.drop(columns=["Seed", "Best Iteration"])
    .agg(["mean", "std", "min", "max"]).T
    .rename_axis("Metric").reset_index()
)

FINAL_SEED = RANDOM_STATE
final_model = models[FINAL_SEED]
final_split = splits[FINAL_SEED]
final_threshold, threshold_analysis = select_threshold(
    final_split["y_thr"], final_split["p_thr"],
    training_pool.loc[final_split["X_thr"].index, "loan_amount"],
    training_pool.loc[final_split["X_thr"].index, "interest_rate"],
)
print(f"\nFinal threshold selected on threshold-tuning split: {final_threshold:.2f}")
print(f"Final model best iteration: {final_model.best_iteration}")
print("Final split row counts:", final_split_sizes)
print("Best iteration is zero-based; selected trees:", final_model.best_iteration + 1)
if final_model.best_iteration + 1 == N_ESTIMATORS:
    print("Best score is at the configured tree limit. This alone does not establish underfitting.")
display(stability_summary)

## 5. Predictive and economic performance

Independent validation was not used for early stopping or threshold selection. Shared-test results are the final reporting metrics. AUC and Gini can be compared with a correctly oriented default model. Here PR-AUC means average precision for **repayment**, so it must not be compared directly with default-class average precision. Accuracy uses the economic approval threshold. Brier score is invariant when both targets and probabilities are complemented.

P&L totals depend on sample size: compare P&L per applicant across differently sized partitions. Reject-all and approve-all provide fixed economic references.

In [ ]:
# Independent internal validation metrics for the seed-42 model.
X_valid = final_split["X_stab"]
y_valid = final_split["y_stab"]
p_valid = final_model.predict_proba(X_valid)[:, 1]
validation_metrics = evaluate_predictions(
    y_valid, p_valid,
    training_pool.loc[X_valid.index, "loan_amount"],
    training_pool.loc[X_valid.index, "interest_rate"],
    final_threshold,
)

X_eval = eval_df[FEATURE_COLS].copy()
for col in [c for c in CAT_COLS if c in FEATURE_COLS]:
    X_eval[col] = X_eval[col].astype(X_valid[col].dtype)
y_eval = eval_df[TARGET].astype(int).copy()
p_eval = final_model.predict_proba(X_eval)[:, 1]
test_metrics = evaluate_predictions(
    y_eval, p_eval, eval_df["loan_amount"], eval_df["interest_rate"], final_threshold
)

metrics_table = pd.DataFrame([
    {"Dataset": "Independent validation", **validation_metrics},
    {"Dataset": "Shared test 20k", **test_metrics},
])

reject_all_pnl = float(profit_vector(
    y_eval, np.zeros(len(y_eval), dtype=bool), eval_df["loan_amount"], eval_df["interest_rate"]
).sum())
approve_all_pnl = float(profit_vector(
    y_eval, np.ones(len(y_eval), dtype=bool), eval_df["loan_amount"], eval_df["interest_rate"]
).sum())
economic_baselines = pd.DataFrame([
    {"Policy": "Reject all", "P&L": reject_all_pnl, "P&L per Applicant": reject_all_pnl / len(y_eval), "Approval Rate": 0.0},
    {"Policy": "Approve all", "P&L": approve_all_pnl, "P&L per Applicant": approve_all_pnl / len(y_eval), "Approval Rate": 1.0},
    {"Policy": "XGBoost threshold", "P&L": test_metrics["P&L"], "P&L per Applicant": test_metrics["P&L per Applicant"], "Approval Rate": test_metrics["Approval Rate"]},
])

print("\nPerformance metrics:")
display(metrics_table.round(5))
print("\nShared-test economic references:")
display(economic_baselines.round(2))

## 6. Validation threshold curve

This plot uses only the threshold-tuning partition. The selected point is saved with the threshold analysis table.

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.plot(threshold_analysis["Threshold"], threshold_analysis["P&L"] / 1_000_000, color="#235789", linewidth=2)
plt.axvline(final_threshold, color="#d1495b", linestyle="--", label=f"Selected threshold = {final_threshold:.2f}")
plt.xlabel("Approval threshold")
plt.ylabel("Validation P&L ($ millions)")
plt.title("Economic threshold selection (validation only)")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "xgboost_threshold_analysis.png", dpi=160, bbox_inches="tight")
plt.show()

## 7. Interpretability — permutation importance

Permutation importance is calculated on the independent seed-42 validation partition, not the shared test. The score is ROC-AUC; a larger decrease after shuffling means the model relies more on that feature for ranking.

In [ ]:
importance_result = permutation_importance(
    final_model,
    X_valid,
    y_valid,
    scoring="roc_auc",
    n_repeats=PERMUTATION_REPEATS,
    random_state=RANDOM_STATE,
    n_jobs=1,  # XGBoost already uses parallel prediction; avoid nested oversubscription.
)
permutation_df = pd.DataFrame({
    "Feature": FEATURE_COLS,
    "Importance Mean": importance_result.importances_mean,
    "Importance Std": importance_result.importances_std,
}).sort_values("Importance Mean", ascending=False)
display(permutation_df.head(20).round(5))

plot_df = permutation_df.head(15).sort_values("Importance Mean")
plt.figure(figsize=(8, 5))
plt.barh(plot_df["Feature"], plot_df["Importance Mean"], xerr=plot_df["Importance Std"], color="#235789", alpha=0.9)
plt.xlabel("Decrease in validation ROC-AUC after permutation")
plt.title("XGBoost permutation importance")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "xgboost_permutation_importance.png", dpi=160, bbox_inches="tight")
plt.show()

## 8. Interpretability — SHAP

Native XGBoost TreeSHAP contributions use exactly the trees selected by early stopping, matching the classifier's predictions. Contributions are in repayment log-odds, not probability points. Their sum plus the base value is checked against the model's raw margin. The beeswarm omits feature-value coloring because native categories have no meaningful numeric order. SHAP plotting requires the optional `shap` package.

In [ ]:
shap_importance_df = pd.DataFrame(columns=["Feature", "Mean Absolute SHAP"])
shap_values_array = None
shap_sample = None

if SHAP_AVAILABLE:
    shap_n = min(SHAP_SAMPLE_SIZE, len(X_valid))
    shap_sample = X_valid.sample(n=shap_n, random_state=RANDOM_STATE)
    selected_rounds = (0, final_model.best_iteration + 1)
    dmatrix = xgb.DMatrix(shap_sample, enable_categorical=True)
    booster = final_model.get_booster()
    contrib = booster.predict(dmatrix, pred_contribs=True, iteration_range=selected_rounds)
    margin = booster.predict(dmatrix, output_margin=True, iteration_range=selected_rounds)
    np.testing.assert_allclose(contrib.sum(axis=1), margin, rtol=1e-4, atol=1e-4)
    shap_values_array = contrib[:, :-1]
    shap_base_values = contrib[:, -1]

    shap_importance_df = pd.DataFrame({
        "Feature": FEATURE_COLS,
        "Mean Absolute SHAP": np.abs(shap_values_array).mean(axis=0),
    }).sort_values("Mean Absolute SHAP", ascending=False)
    display(shap_importance_df.head(20).round(5))

    plt.figure()
    shap.summary_plot(shap_values_array, feature_names=FEATURE_COLS, show=False, max_display=20)
    plt.title("XGBoost SHAP beeswarm — independent validation sample")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "xgboost_shap_beeswarm.png", dpi=160, bbox_inches="tight")
    plt.show()

    # Local explanation for the first sampled validation record.
    local_idx = 0
    local_exp = shap.Explanation(
        values=shap_values_array[local_idx],
        base_values=float(np.asarray(shap_base_values).reshape(-1)[local_idx]) if np.asarray(shap_base_values).ndim else float(shap_base_values),
        data=None,
        feature_names=FEATURE_COLS,
    )
    shap.plots.waterfall(local_exp, max_display=15, show=False)
    plt.title("Local SHAP contributions — validation record")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "xgboost_shap_local_waterfall.png", dpi=160, bbox_inches="tight")
    plt.show()

    local_shap_df = pd.DataFrame({
        "Feature": FEATURE_COLS,
        "SHAP Value": shap_values_array[local_idx],
        "Absolute SHAP Value": np.abs(shap_values_array[local_idx]),
    }).sort_values("Absolute SHAP Value", ascending=False)
    print("Local record ID:", training_pool.loc[shap_sample.index[local_idx], ID_COL])
    display(local_shap_df.head(15).round(5))
else:
    print("SHAP is unavailable; permutation importance was still produced. Import error:", SHAP_IMPORT_ERROR)

## 9. Fairness — shared-test subgroup results

All four audit attributes are evaluated, including employment and education when they are predictors. The positive outcome is repayment and the positive decision is approval: **TPR = approval rate among repayers; FPR = approval rate among defaulters**. For the white-box notebook's rejection/default convention, repayment TPR = 1 − white-box FPR, and repayment FPR = 1 − white-box TPR.

Rate gaps are stored as proportions; multiply by 100 to report percentage points. Group counts and outcome denominators help identify noisy small-group rates. These are descriptive group disparities, not a blanket fairness certification.

In [ ]:
def fairness_by_group(df, y_true, probabilities, threshold, sensitive_col):
    decision = (np.asarray(probabilities) >= threshold).astype(int)
    y_arr = np.asarray(y_true, dtype=int)
    rows = []
    group_values = df[sensitive_col].astype("object").fillna("<missing>").to_numpy()
    for group_name in pd.unique(group_values):
        mask = group_values == group_name
        yg, dg, pg = y_arr[mask], decision[mask], np.asarray(probabilities)[mask]
        positive = yg == 1
        negative = yg == 0
        rows.append({
            "Audit Attribute": sensitive_col,
            "Group": str(group_name),
            "N": int(mask.sum()),
            "Approved": int(dg.sum()),
            "Repaid N": int(positive.sum()),
            "Default N": int(negative.sum()),
            "Approval Rate": float(dg.mean()),
            "TPR": float(dg[positive].mean()) if positive.sum() else np.nan,
            "FPR": float(dg[negative].mean()) if negative.sum() else np.nan,
            "Precision": float(yg[dg == 1].mean()) if dg.sum() else np.nan,
            "AUC": float(roc_auc_score(yg, pg)) if len(np.unique(yg)) == 2 else np.nan,
        })
    return pd.DataFrame(rows)

fairness_details = pd.concat([
    fairness_by_group(eval_df, y_eval, p_eval, final_threshold, col)
    for col in FAIRNESS_ATTRIBUTES
], ignore_index=True)

fairness_rows = []
for attr, groups in fairness_details.groupby("Audit Attribute", sort=False):
    fairness_rows.append({
        "Audit Attribute": attr,
        "Demographic Parity Gap": groups["Approval Rate"].max() - groups["Approval Rate"].min(),
        "Equal Opportunity Gap": groups["TPR"].max() - groups["TPR"].min(),
        "FPR Gap": groups["FPR"].max() - groups["FPR"].min(),
        "Equalized Odds Gap": max(groups["TPR"].max() - groups["TPR"].min(), groups["FPR"].max() - groups["FPR"].min()),
        "Disparate Impact Ratio": groups["Approval Rate"].min() / groups["Approval Rate"].max() if groups["Approval Rate"].max() > 0 else np.nan,
    })
fairness_summary = pd.DataFrame(fairness_rows)

display(fairness_details.round(4))
display(fairness_summary.round(4))

## 10. Approval-rate equivalence and feature associations

TOST tests the approval-rate difference against the prespecified ±2 percentage-point margin. Holm adjustment uses one family across all reported attribute/group comparisons. The margin is a project assumption, not a legal standard. Unadjusted and adjusted p-values are both exported; compare like-for-like when discussing the white-box results.

The normal approximation is not used when either group's observed approval or rejection count is below five; those comparisons are marked not assessed. Groups smaller than the configured minimum are omitted. Failure to establish equivalence does not prove inequivalence.

Association checks use the training pool and the current predictors. A variable's association with itself is excluded. Direct inclusion of an audit attribute is documented in the configuration. Low univariate associations do not rule out nonlinear or multivariate proxy information.

In [ ]:
from scipy.stats import chi2_contingency, norm


def tost_two_proportions(approved_a, n_a, approved_b, n_b, delta=TOST_DELTA):
    """Two one-sided z tests for H1: -delta < p_a - p_b < delta."""
    p_a, p_b = approved_a / n_a, approved_b / n_b
    diff = p_a - p_b
    se = np.sqrt(p_a * (1 - p_a) / n_a + p_b * (1 - p_b) / n_b)
    # The Wald approximation is unreliable with sparse successes/failures.
    # Do not certify equivalence for those pairs using a zero estimated variance.
    counts = [approved_a, n_a - approved_a, approved_b, n_b - approved_b]
    if min(counts) < 5 or se == 0:
        return float(diff), np.nan
    else:
        z_lower = (diff + delta) / se   # H1: diff > -delta
        z_upper = (diff - delta) / se   # H1: diff < +delta
        p_lower = norm.sf(z_lower)
        p_upper = norm.cdf(z_upper)
    return float(diff), float(max(p_lower, p_upper))


def holm_adjust(p_values):
    """Holm family-wise adjusted p-values, returned in original order."""
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    m = len(p_values)
    adjusted_sorted = np.maximum.accumulate((m - np.arange(m)) * p_values[order])
    adjusted_sorted = np.minimum(adjusted_sorted, 1.0)
    adjusted = np.empty(m, dtype=float)
    adjusted[order] = adjusted_sorted
    return adjusted


tost_rows = []
for attr, groups in fairness_details.groupby("Audit Attribute", sort=False):
    eligible = groups.loc[groups["N"] >= MIN_GROUP_N_FOR_TOST].reset_index(drop=True)
    for i, j in combinations(range(len(eligible)), 2):
        a, b = eligible.iloc[i], eligible.iloc[j]
        diff, p_value = tost_two_proportions(a["Approved"], a["N"], b["Approved"], b["N"])
        tost_rows.append({
            "Audit Attribute": attr,
            "Group A": a["Group"], "Group B": b["Group"],
            "N A": int(a["N"]), "N B": int(b["N"]),
            "Approval Rate Difference": diff,
            "Delta": TOST_DELTA,
            "TOST p-value": p_value,
        })

tost_results = pd.DataFrame(tost_rows)
if not tost_results.empty:
    tost_results["Holm Adjusted p-value"] = holm_adjust(tost_results["TOST p-value"].fillna(1.0))
    tost_results.loc[tost_results["TOST p-value"].isna(), "Holm Adjusted p-value"] = np.nan
    tost_results["Equivalent within delta"] = tost_results["Holm Adjusted p-value"] < TOST_ALPHA
    tost_results["Interpretation"] = np.where(
        tost_results["Equivalent within delta"],
        "Evidence supports equivalence within the specified margin",
        "Equivalence not established",
    )
    tost_results.loc[tost_results["TOST p-value"].isna(), "Interpretation"] = "Not assessed: sparse approval/rejection counts"
display(tost_results.round(5))


def cramers_v(a, b):
    table = pd.crosstab(a, b)
    if min(table.shape) < 2:
        return np.nan
    chi2 = chi2_contingency(table, correction=False)[0]
    n = table.to_numpy().sum()
    return float(np.sqrt(chi2 / (n * (min(table.shape) - 1)))) if n else np.nan


def eta_squared(category, numeric):
    data = pd.DataFrame({"category": category, "numeric": numeric}).dropna()
    if len(data) == 0 or data["numeric"].var() == 0:
        return np.nan
    grand_mean = data["numeric"].mean()
    ss_between = sum(len(g) * (g.mean() - grand_mean) ** 2 for _, g in data.groupby("category", observed=True)["numeric"])
    ss_total = ((data["numeric"] - grand_mean) ** 2).sum()
    return float(ss_between / ss_total) if ss_total else np.nan


proxy_rows = []
proxy_cat_candidates = [c for c in FEATURE_COLS if c in CAT_COLS]
proxy_num_candidates = [c for c in FEATURE_COLS if c not in CAT_COLS and pd.api.types.is_numeric_dtype(training_pool[c])]
for protected in FAIRNESS_ATTRIBUTES:
    for candidate in proxy_cat_candidates:
        if candidate == protected:
            continue  # Self-association is direct inclusion, not a proxy relationship.
        proxy_rows.append({
            "Audit Attribute": protected,
            "Candidate Feature": candidate,
            "Association Measure": "Cramer's V",
            "Association Value": cramers_v(training_pool[protected], training_pool[candidate]),
        })
    for candidate in proxy_num_candidates:
        proxy_rows.append({
            "Audit Attribute": protected,
            "Candidate Feature": candidate,
            "Association Measure": "Eta-squared",
            "Association Value": eta_squared(training_pool[protected], training_pool[candidate]),
        })
proxy_checks = pd.DataFrame(proxy_rows)
print("Univariate association checks only; they do not rule out proxy effects.")
display(proxy_checks.sort_values("Association Value", ascending=False).round(5))

## 11. Export and run provenance

Artifacts are saved in the exact run directory printed at startup. Feature modes, fast/full mode and run IDs have separate directories. The configuration records predictor names, fit-derived category vocabularies, actual partition sizes, selected trees, target/decision definitions, package versions and economic assumptions. The split manifest records the final seed's development IDs. Existing run directories retain their previous results.

In [ ]:
# Final shared-test decisions use the threshold chosen on the separate threshold split.
test_decisions = (p_eval >= final_threshold).astype(int)
prediction_output = pd.DataFrame({
    ID_COL: eval_df[ID_COL].to_numpy(),
    "Actual": y_eval.to_numpy(),
    "Predicted_Probability": p_eval,
    "Approval_Threshold": final_threshold,
    "Decision": np.where(test_decisions == 1, "Approve", "Reject"),
})

final_summary = pd.DataFrame([{
    "Model": "XGBoost",
    "Final Seed": FINAL_SEED,
    "Validation AUC": validation_metrics["ROC-AUC"],
    "Shared Test AUC": test_metrics["ROC-AUC"],
    "Shared Test PR-AUC": test_metrics["PR-AUC"],
    "Shared Test Gini": test_metrics["Gini"],
    "Shared Test KS": test_metrics["KS"],
    "Shared Test Brier Score": test_metrics["Brier Score"],
    "Shared Test PCC / Accuracy": test_metrics["PCC / Accuracy"],
    "Economic Threshold": final_threshold,
    "Shared Test P&L": test_metrics["P&L"],
    "P&L per Applicant": test_metrics["P&L per Applicant"],
    "Approval Rate": test_metrics["Approval Rate"],
    "Default Rate Among Approved": test_metrics["Default Rate Among Approved"],
    "Optional Predictors": ", ".join(OPTIONAL_PREDICTORS) if OPTIONAL_PREDICTORS else "None",
    "Employment/Education Predictors Included": INCLUDE_EMPLOYMENT_AND_EDUCATION,
    "Fast Mode": FAST_MODE,
}])

config = {
    "model": "XGBoost",
    "run_id": RUN_ID,
    "run_label": RUN_LABEL,
    "output_directory": str(OUTPUT_DIR),
    "actual_split_sizes": final_split_sizes,
    "selected_trees": int(final_model.best_iteration + 1),
    "model_parameters": final_model.get_params(),
    "category_levels": {c: X_valid[c].cat.categories.tolist() for c in CAT_COLS if c in FEATURE_COLS},
    "package_versions": {name: version(name) for name in ["numpy", "pandas", "scikit-learn", "xgboost", "scipy"]},
    "probability_event": "repayment",
    "approval_rule": "P(repaid) >= threshold",
    "fairness_tpr": "P(approve | repaid)",
    "fairness_fpr": "P(approve | default)",
    "shap_available": SHAP_AVAILABLE,
    "target": TARGET,
    "target_mapping": {"1": "repaid", "0": "default"},
    "shared_test_path": str(EVAL_PATH),
    "shared_test_rows": int(len(eval_df)),
    "training_pool_rows": int(len(training_pool)),
    "fit_fraction": FIT_FRACTION,
    "early_stop_fraction": EARLY_STOP_FRACTION,
    "threshold_fraction": THRESHOLD_FRACTION,
    "stability_fraction": STABILITY_FRACTION,
    "final_seed": FINAL_SEED,
    "stability_seeds": STABILITY_SEEDS,
    "fast_mode": FAST_MODE,
    "features": FEATURE_COLS,
    "fairness_audit_attributes": FAIRNESS_ATTRIBUTES,
    "include_employment_and_education": INCLUDE_EMPLOYMENT_AND_EDUCATION,
    "optional_predictors": OPTIONAL_PREDICTORS,
    "use_class_weight": USE_CLASS_WEIGHT,
    "threshold": final_threshold,
    "tost_delta": TOST_DELTA,
    "tost_alpha": TOST_ALPHA,
    "pnl_rule": {
        "approved_and_repaid": "+loan_amount * interest_rate / 100",
        "approved_and_defaulted": "-loan_amount * 0.60",
        "rejected": 0,
    },
}

# Save both the portable XGBoost model and a Python joblib copy.
final_model.save_model(str(OUTPUT_DIR / "xgboost_final_model.json"))
joblib.dump(final_model, OUTPUT_DIR / "xgboost_final_model.joblib")
with open(OUTPUT_DIR / "xgboost_final_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

final_split_ids.to_csv(OUTPUT_DIR / "xgboost_development_split_ids.csv", index=False)
metrics_table.to_csv(OUTPUT_DIR / "xgboost_metrics.csv", index=False)
final_summary.to_csv(OUTPUT_DIR / "xgboost_final_summary.csv", index=False)
economic_baselines.to_csv(OUTPUT_DIR / "xgboost_economic_baselines.csv", index=False)
threshold_analysis.to_csv(OUTPUT_DIR / "xgboost_threshold_analysis.csv", index=False)
stability_runs.to_csv(OUTPUT_DIR / "xgboost_stability_runs.csv", index=False)
stability_summary.to_csv(OUTPUT_DIR / "xgboost_stability_summary.csv", index=False)
permutation_df.to_csv(OUTPUT_DIR / "xgboost_permutation_importance.csv", index=False)
if SHAP_AVAILABLE and not shap_importance_df.empty:
    shap_importance_df.to_csv(OUTPUT_DIR / "xgboost_shap_global_importance.csv", index=False)
    local_shap_df.to_csv(OUTPUT_DIR / "xgboost_shap_local_explanation.csv", index=False)
fairness_details.to_csv(OUTPUT_DIR / "xgboost_fairness_details.csv", index=False)
fairness_summary.to_csv(OUTPUT_DIR / "xgboost_fairness_summary.csv", index=False)
tost_results.to_csv(OUTPUT_DIR / "xgboost_fairness_tost.csv", index=False)
proxy_checks.to_csv(OUTPUT_DIR / "xgboost_proxy_checks.csv", index=False)
prediction_output.to_csv(OUTPUT_DIR / "xgboost_shared_test_predictions.csv", index=False)

print("\nSaved XGBoost artifacts to:", OUTPUT_DIR.resolve())
display(final_summary.round(5))

## Reporting checklist

- State the run ID, feature switch and actual predictors with every reported result. Compare model families using agreed raw predictors, development partitions, target conventions and economic rules.
- Report threshold as a repayment-probability cutoff. Threshold selection occurs only on its reserved development partition.
- Report both predictive quality and economic outcomes; AUC gains are not guaranteed by adding features.
- Stability here varies the split and model seed together. Its standard deviation is descriptive across five runs, not a confidence interval for shared-test AUC.
- Discuss subgroup sample sizes, approval gaps and error-rate gaps together. TOST addresses approval-rate equivalence for the chosen margin and comparison family only.
- Use feature comparisons to document trade-offs. Any further model selection belongs on development data, not repeated optimization against the shared test.
- Fast-mode results are development checks. Restart the kernel and run all cells to obtain results for a changed feature setting.